# 04 — Generated Route Evaluation

## Why evaluate generated routes?

In language modeling, we evaluate generated text using metrics like BLEU, ROUGE, or perplexity. For climbing routes, we need domain-specific evaluation:

1. **Validity**: Does the route follow the rules of climbing boards?
2. **Novelty**: Is the route different from existing climbs, or just a copy?
3. **Geometric plausibility**: Are the holds in reasonable positions?
4. **Grade consistency**: Does the route's predicted grade match the requested grade?

### Validity checks

A "basic valid" route must have:
- At least 3 holds
- No duplicate placements
- At least one start hold and one finish hold
- All holds from the same board (no mixing TB2 and Kilter holds)

A "strict valid" route additionally has:
- At least one middle hold (most real climbs have more than just start + finish)
- At least 4 holds total

### Novelty metrics

We measure novelty using **Jaccard distance**: 1 minus the Jaccard similarity between the generated route's hold set and the most similar real route's hold set.

- Jaccard similarity = |A intersection B| / |A union B|
- Novelty distance = 1 - Jaccard similarity

A novelty distance of 1.0 means the generated route shares no holds with any real route. A distance of 0.0 means it's identical to an existing route.



In [ ]:
from __future__ import annotations

import ast
import re
from pathlib import Path
from typing import Iterable

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from scipy.spatial.distance import pdist

ROOT = Path.cwd().resolve()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent

In [ ]:
# Load generated routes and real routes for comparison
# NOTE: This notebook requires that you've run notebook 03 first to
# generate and save the routes.

TOKENIZED = ROOT / "data" / "processed" / "tokenized"
GENERATED = ROOT / "data" / "processed" / "generation"

# Check if required files exist
generated_path = GENERATED / "generated_routes.csv"
routes_path = TOKENIZED / "route_sequences.csv"
token_meta_path = TOKENIZED / "token_metadata.csv"

if not generated_path.exists():
    raise FileNotFoundError(
        f"Generated routes not found at: {generated_path}\n"
        f"Please run notebook 03 first to generate and save routes,\n"
        f"or run: python scripts/03_train_route_generator.py"
    )

if not routes_path.exists() or not token_meta_path.exists():
    raise FileNotFoundError(
        f"Tokenized data not found at: {TOKENIZED}\n"
        f"Please run notebook 01 first to tokenize routes,\n"
        f"or run: python scripts/01_tokenize_routes.py"
    )

df_generated = pd.read_csv(generated_path)
df_real = pd.read_csv(routes_path)
df_token_meta = pd.read_csv(token_meta_path)

print(f"Generated routes: {len(df_generated):,}")
print(f"Real routes: {len(df_real):,}")

### Token parsing and validity helpers

In [ ]:
# Parse generated token strings and compute basic route-validity flags.
HOLD_TOKEN_PATTERN = re.compile(r"^<([A-Z0-9_]+)_p(\d+)_(start|middle|finish|foot|unknown)>$")

def parse_tokens(value) -> list[str]:
    """Parse tokens from a list, repr-style list string, or whitespace sequence."""
    if isinstance(value, list):
        return [str(v) for v in value]
    if not isinstance(value, str):
        return []

    try:
        parsed = ast.literal_eval(value)
        if isinstance(parsed, list):
            return [str(v) for v in parsed]
    except (SyntaxError, ValueError):
        pass

    return value.split()

def tokens_to_hold_records(tokens: Iterable[str]) -> list[dict[str, object]]:
    """Extract hold records from model tokens using the shared hold-token grammar."""
    rows: list[dict[str, object]] = []
    for token in tokens:
        match = HOLD_TOKEN_PATTERN.match(str(token))
        if match is None:
            continue
        board_prefix = match.group(1)
        rows.append(
            {
                "token": str(token),
                "board_token_prefix": board_prefix,
                "board_prefix": board_prefix,
                "placement_id": int(match.group(2)),
                "role": match.group(3),
            }
        )
    return rows

def parse_token_list(value) -> list[str]:
    """Compatibility wrapper around the shared token parser."""
    return parse_tokens(value)

def validity_from_records(records: list[dict[str, object]], requested_board_prefix: str | None = None) -> dict[str, object]:
    """Compute evaluation-specific route-validity flags from hold records."""
    placements = [int(record["placement_id"]) for record in records]
    roles = [str(record["role"]) for record in records]
    prefixes = [str(record["board_token_prefix"]) for record in records]
    one_board_only = len(set(prefixes)) <= 1
    matches_requested_board = requested_board_prefix is None or all(prefix == requested_board_prefix for prefix in prefixes)

    out = {
        "n_holds_eval": len(records),
        "n_unique_placements_eval": len(set(placements)),
        "has_duplicate_placements_eval": len(records) != len(set(placements)),
        "one_board_only_eval": one_board_only,
        "matches_requested_board_eval": matches_requested_board,
        "n_start_eval": roles.count("start"),
        "n_middle_eval": roles.count("middle"),
        "n_foot_eval": roles.count("foot"),
        "n_finish_eval": roles.count("finish"),
        "has_start_eval": "start" in roles,
        "has_middle_eval": "middle" in roles,
        "has_finish_eval": "finish" in roles,
    }
    out["basic_valid_eval"] = (
        one_board_only
        and out["n_holds_eval"] >= 3
        and out["n_holds_eval"] == out["n_unique_placements_eval"]
        and out["has_start_eval"]
        and out["has_finish_eval"]
    )
    out["strict_valid_eval"] = (
        out["basic_valid_eval"]
        and out["has_middle_eval"]
        and out["n_holds_eval"] >= 4
    )
    return out

## Parse generated tokens and check validity

We parse the generated token sequences and check each route for validity.



In [ ]:
# Parse the token strings into structured records
df_generated["tokens_parsed"] = df_generated["tokens"].apply(parse_token_list)

# Extract hold information from tokens
df_generated["hold_records"] = df_generated["tokens_parsed"].apply(tokens_to_hold_records)

# Check validity for each generated route
validity = pd.DataFrame(df_generated["hold_records"].apply(validity_from_records).tolist())
df_eval = pd.concat([df_generated.reset_index(drop=True), validity], axis=1)

print("Validity rates by board:")
print("=" * 50)
validity_summary = df_eval.groupby("board_key").agg(
    total=("basic_valid_eval", "count"),
    basic_valid_rate=("basic_valid_eval", "mean"),
    strict_valid_rate=("strict_valid_eval", "mean"),
    avg_holds=("n_holds_eval", "mean"),
).round(3)
print(validity_summary)

### Novelty helpers

In [ ]:
# Compare generated hold sets to real routes using Jaccard similarity.
def frames_to_holds(frames: str | None) -> list[tuple[int, int]]:
    """Parse a frames string into ``(placement_id, role_id)`` pairs."""
    if not isinstance(frames, str):
        return []
    return [(int(p), int(r)) for p, r in re.findall(r"p(\d+)r(\d+)", frames)]

def holds_to_placement_set(holds: Iterable[tuple[int, int]]) -> frozenset[int]:
    """Drop role IDs and represent a route by its unique placement IDs."""
    return frozenset(int(placement_id) for placement_id, _ in holds)

def jaccard(a: frozenset[int], b: frozenset[int]) -> float:
    """Return Jaccard similarity between two placement sets."""
    if not a and not b:
        return 1.0
    if not a or not b:
        return 0.0
    return len(a & b) / len(a | b)

def nearest_real_route_same_board(
    generated_set: frozenset[int],
    generated_board_key: str,
    real_df: pd.DataFrame,
) -> dict[str, object]:
    """Find the most similar real route on the same board by Jaccard score.

    .. note::

       This function performs an O(n) linear scan over all real routes for
       the matching board, computing a Jaccard similarity for each one. With
       ~256K training examples, evaluating 400 generated routes costs roughly
       O(100M) Jaccard comparisons. This is acceptable for evaluation scripts
       but would not scale to a real-time or high-throughput setting without
       an approximate nearest-neighbour index.
    """
    board_frame = real_df[real_df["board_key"] == generated_board_key]
    if board_frame.empty:
        return {
            "nearest_real_jaccard": np.nan,
            "nearest_real_uuid": None,
            "nearest_real_name": None,
            "nearest_real_grouped_v": None,
            "nearest_real_angle": None,
            "novelty_distance": np.nan,
        }

    similarities = board_frame["hold_set"].map(lambda hold_set: jaccard(generated_set, hold_set))
    best_idx = similarities.idxmax()
    row = board_frame.loc[best_idx]

    nearest_real_jaccard = float(similarities.loc[best_idx])
    return {
        "nearest_real_jaccard": nearest_real_jaccard,
        "nearest_real_uuid": row["uuid"],
        "nearest_real_name": row["climb_name"],
        "nearest_real_grouped_v": row["grouped_v"],
        "nearest_real_angle": row["angle"],
        "novelty_distance": 1.0 - nearest_real_jaccard,
    }

## Novelty against real climbs

For each generated route, we find the most similar real route from the same board (by Jaccard similarity of hold sets). A good generator should produce routes that are novel (low Jaccard similarity to existing routes) while still being valid.



In [ ]:
# Convert hold sets to frozensets for fast comparison
df_eval["hold_set"] = df_eval["hold_records"].apply(
    lambda records: frozenset(int(record["placement_id"]) for record in records)
)

# Parse real routes' frames strings into hold sets
df_real["real_holds"] = df_real["frames"].apply(frames_to_holds)
df_real["hold_set"] = df_real["real_holds"].apply(holds_to_placement_set)

# Find nearest real route for each generated route
print("Computing novelty (finding nearest real route for each generated route)...")
print("This may take a few minutes...")

nearest = pd.DataFrame(
    df_eval.apply(
        lambda row: nearest_real_route_same_board(
            generated_set=row["hold_set"],
            generated_board_key=row["board_key"],
            real_df=df_real,
        ),
        axis=1,
    ).tolist()
)
df_eval = pd.concat([df_eval, nearest], axis=1)

print("\nNovelty statistics by board:")
print("=" * 50)
novelty_summary = df_eval.groupby("board_key").agg(
    mean_jaccard=("nearest_real_jaccard", "mean"),
    mean_novelty=("novelty_distance", "mean"),
    median_novelty=("novelty_distance", "median"),
).round(3)
print(novelty_summary)

### Geometry helpers

In [ ]:
# Compute simple geometric descriptors from placement coordinates.
def build_placement_coords(df_token_meta: pd.DataFrame) -> dict[tuple[str, int], dict[str, float]]:
    """Build a placement-coordinate lookup from token metadata."""
    hold_meta = df_token_meta[df_token_meta["kind"] == "hold"].dropna(subset=["placement_id"]).copy()
    coords = {}
    for _, row in hold_meta.drop_duplicates(["board_key", "placement_id"]).iterrows():
        key = (str(row["board_key"]), int(row["placement_id"]))
        coords[key] = {
            "x": float(row["x"]),
            "y": float(row["y"]),
        }
    return coords

def simple_route_features(
    board_key: str,
    records: list[dict[str, object]],
    placement_coords: dict[tuple[str, int], dict[str, float]],
) -> dict[str, float]:
    """Compute simple geometric route features from hold coordinates.

    These features are descriptive rather than a full climbing-physics model:
    height/width describe route spread, and hand-reach distances summarize the
    pairwise spacing among start/middle/finish holds.
    """
    rows = []
    for record in records:
        key = (str(board_key), int(record["placement_id"]))
        coord = placement_coords.get(key)
        if coord is None:
            continue
        x = float(coord["x"])
        y = float(coord["y"])
        if np.isnan(x) or np.isnan(y):
            continue
        role = str(record["role"])
        rows.append(
            {
                "x": x,
                "y": y,
                "role": role,
                "is_hand": role in {"start", "middle", "finish"},
                "is_foot": role == "foot",
            }
        )

    if not rows:
        return {
            "geom_n_holds": 0.0,
            "geom_height": np.nan,
            "geom_width": np.nan,
            "geom_mean_y": np.nan,
            "geom_mean_x_abs": np.nan,
            "geom_mean_hand_reach": np.nan,
            "geom_max_hand_reach": np.nan,
        }

    d = pd.DataFrame(rows)
    out = {
        "geom_n_holds": float(len(d)),
        "geom_height": float(d["y"].max() - d["y"].min()),
        "geom_width": float(d["x"].max() - d["x"].min()),
        "geom_mean_y": float(d["y"].mean()),
        "geom_mean_x_abs": float(d["x"].abs().mean()),
    }

    hands = d[d["is_hand"]].sort_values(["y", "x"])
    if len(hands) >= 2:
        distances = pdist(hands[["x", "y"]].values)
        out["geom_mean_hand_reach"] = float(distances.mean())
        out["geom_max_hand_reach"] = float(distances.max())
    else:
        out["geom_mean_hand_reach"] = np.nan
        out["geom_max_hand_reach"] = np.nan

    return out

## Geometric descriptors

We compute simple geometric features for each generated route:

- `geom_n_holds`: Number of holds
- `geom_height`: Vertical extent of the route
- `geom_width`: Horizontal extent
- `geom_mean_hand_reach`: Average distance between hand holds

These features help us understand whether generated routes have reasonable spatial properties.



In [ ]:
# Build coordinate lookup from token metadata
coords = build_placement_coords(df_token_meta)

# Compute geometric features for each generated route
geom = pd.DataFrame(
    df_eval.apply(
        lambda row: simple_route_features(
            board_key=row["board_key"],
            records=row["hold_records"],
            placement_coords=coords,
        ),
        axis=1,
    ).tolist()
)
df_eval = pd.concat([df_eval, geom], axis=1)

print("Geometric feature statistics by board:")
print("=" * 50)
geom_summary = df_eval.groupby("board_key").agg(
    mean_holds=("geom_n_holds", "mean"),
    mean_height=("geom_height", "mean"),
    mean_width=("geom_width", "mean"),
    mean_hand_reach=("geom_mean_hand_reach", "mean"),
).round(3)
print(geom_summary)

### Critic model and grade helpers

In [ ]:
# Map BoardLib display difficulties into grouped V-grade tokens.
GRADE_TO_V = {
    10: 0, 11: 0, 12: 0,
    13: 1, 14: 1,
    15: 2,
    16: 3, 17: 3,
    18: 4, 19: 4,
    20: 5, 21: 5,
    22: 6,
    23: 7,
    24: 8, 25: 8,
    26: 9,
    27: 10,
    28: 11,
    29: 12,
    30: 13,
    31: 14,
    32: 15,
    33: 16,
}

def to_grouped_v(display_difficulty: float) -> int:
    """Map a continuous display difficulty to the nearest grouped V grade."""
    rounded = int(round(float(display_difficulty)))
    rounded = max(min(rounded, max(GRADE_TO_V)), min(GRADE_TO_V))
    return GRADE_TO_V[rounded]

def grade_token(display_difficulty: float) -> str:
    """Return the grade-conditioning token for a display difficulty value."""
    return f"<GRADE_V{to_grouped_v(display_difficulty)}>"

# Transformer encoder used as a continuous grade regressor.
class JointRouteTransformerRegressor(nn.Module):
    """Transformer encoder for joint TB2/Kilter route difficulty prediction.

    Inputs are token IDs plus an attention mask. Token, position, and learned
    projections of coordinate metadata are added before the encoder. The first
    ``<CLS>`` position is then used as a pooled route representation for scalar
    difficulty regression.
    """

    def __init__(
        self,
        vocab_size: int,
        max_len: int,
        coord_features: torch.Tensor,
        d_model: int = 128,
        nhead: int = 4,
        num_layers: int = 4,
        dim_feedforward: int = 256,
        dropout: float = 0.10,
        pad_id: int = 0,
    ):
        """Create the encoder, coordinate projection, and regression head."""
        super().__init__()
        self.vocab_size = vocab_size
        self.max_len = max_len
        self.d_model = d_model
        self.pad_id = pad_id

        self.token_emb = nn.Embedding(vocab_size, d_model, padding_idx=pad_id)
        self.pos_emb = nn.Embedding(max_len, d_model)

        self.register_buffer("coord_features", coord_features.clone().float())
        self.coord_proj = nn.Linear(coord_features.shape[1], d_model)

        encoder_layer = nn.TransformerEncoderLayer(
            d_model=d_model,
            nhead=nhead,
            dim_feedforward=dim_feedforward,
            dropout=dropout,
            activation="gelu",
            batch_first=True,
            norm_first=True,
        )
        self.encoder = nn.TransformerEncoder(
            encoder_layer,
            num_layers=num_layers,
            enable_nested_tensor=False,
        )
        self.norm = nn.LayerNorm(d_model)
        self.head = nn.Sequential(
            nn.Linear(d_model, d_model),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(d_model, 1),
        )

    def forward(self, input_ids: torch.Tensor, attention_mask: torch.Tensor) -> torch.Tensor:
        """Return one continuous difficulty prediction per input sequence."""
        batch_size, seq_len = input_ids.shape
        positions = torch.arange(seq_len, device=input_ids.device).unsqueeze(0).expand(batch_size, seq_len)

        # Coordinate features are indexed by token ID, so every occurrence of a
        # hold token gets the same physical x/y hint wherever it appears.
        x = self.token_emb(input_ids) + self.pos_emb(positions)
        x = x + self.coord_proj(self.coord_features[input_ids])

        key_padding_mask = ~attention_mask.bool()
        h = self.encoder(x, src_key_padding_mask=key_padding_mask)
        h = self.norm(h)

        cls_state = h[:, 0, :]
        return self.head(cls_state).squeeze(-1)

## Grade consistency (using the trained critic)

If we have a trained grade predictor (from notebook 02), we can use it as a **critic** to check whether generated routes have grades consistent with what was requested.


In [ ]:
# Try to load the grade critic from notebook 02
GRADE_MODEL_PATH = ROOT / "models" / "joint_transformer_grade_predictor.pth"

def load_grade_critic(model_path, device):
    """Load the trained grade predictor model."""
    if not model_path.exists():
        return None
    try:
        checkpoint = torch.load(model_path, map_location=device, weights_only=False)
    except TypeError:
        checkpoint = torch.load(model_path, map_location=device)

    cfg = checkpoint["config"]
    stoi = {str(k): int(v) for k, v in checkpoint["stoi"].items()}
    coord_features = checkpoint["coord_features"]
    if not isinstance(coord_features, torch.Tensor):
        coord_features = torch.tensor(coord_features, dtype=torch.float32)

    model = JointRouteTransformerRegressor(
        vocab_size=cfg["vocab_size"],
        max_len=cfg["max_len"],
        coord_features=coord_features,
        d_model=cfg.get("d_model", 128),
        nhead=cfg.get("nhead", 4),
        num_layers=cfg.get("num_layers", 4),
        dim_feedforward=cfg.get("dim_feedforward", 256),
        dropout=cfg.get("dropout", 0.10),
        pad_id=cfg.get("pad_id", stoi["<PAD>"]),
    ).to(device)
    model.load_state_dict(checkpoint["model_state_dict"])
    model.eval()

    return {
        "model": model,
        "stoi": stoi,
        "pad_id": stoi["<PAD>"],
        "unk_id": stoi["<UNK>"],
        "max_len": cfg["max_len"],
    }


def predict_generated_grade(tokens, critic, device):
    """Predict the difficulty of a generated route using the critic."""
    model = critic["model"]
    stoi = critic["stoi"]
    pad_id = critic["pad_id"]
    unk_id = critic["unk_id"]
    max_len = critic["max_len"]

    # Remove grade tokens and replace BOS with CLS
    tokens = [t for t in tokens if not t.startswith("<GRADE_")]
    if tokens and tokens[0] == "<BOS>":
        tokens = ["<CLS>"] + tokens[1:]
    else:
        tokens = ["<CLS>"] + tokens

    ids = [stoi.get(t, unk_id) for t in tokens][:max_len]
    mask = [1] * len(ids)
    if len(ids) < max_len:
        pad_n = max_len - len(ids)
        ids += [pad_id] * pad_n
        mask += [0] * pad_n

    with torch.no_grad():
        input_ids = torch.tensor([ids], dtype=torch.long, device=device)
        attention_mask = torch.tensor([mask], dtype=torch.bool, device=device)
        return float(model(input_ids, attention_mask).cpu().item())


device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
critic = load_grade_critic(GRADE_MODEL_PATH, device)

if critic is not None:
    print("Grade critic loaded successfully!")
    print(f"Device: {device}")
else:
    print("No trained grade critic found. Skipping critic-based scoring.")
    print("Run notebook 02 first to train the grade predictor.")

In [ ]:
# Apply the critic to evaluate grade consistency
if critic is not None:
    df_eval["critic_pred_display_difficulty"] = df_eval["tokens_parsed"].apply(
        lambda tokens: predict_generated_grade(tokens, critic, device)
    )
    df_eval["critic_pred_grouped_v"] = df_eval["critic_pred_display_difficulty"].apply(to_grouped_v)
    df_eval["critic_v_error"] = df_eval["critic_pred_grouped_v"] - df_eval["requested_grouped_v"]

    print("Grade consistency by board:")
    print("=" * 50)
    critic_summary = df_eval.groupby("board_key").agg(
        exact_v=("critic_v_error", lambda s: float((s == 0).mean() * 100)),
        within_1_v=("critic_v_error", lambda s: float((s.abs() <= 1).mean() * 100)),
        within_2_v=("critic_v_error", lambda s: float((s.abs() <= 2).mean() * 100)),
        mean_error=("critic_v_error", "mean"),
    ).round(2)
    print(critic_summary)
else:
    print("Skipping critic evaluation (no model loaded).")

## Ranking generated routes

We rank candidates by a composite score that rewards:
- **Basic validity** (required): At least 3 holds, start/finish, no duplicates, one board
- **Strict validity** (bonus): Also has middle holds and 4+ holds
- **Novelty** (higher is better): Distance from nearest real route
- **Grade consistency** (if critic available): Predicted grade close to requested grade



In [ ]:
# Rank candidates by composite score
ranked = df_eval.copy()
ranked["score"] = 0.0
ranked["score"] += ranked["basic_valid_eval"].astype(float) * 2.0
ranked["score"] += ranked["strict_valid_eval"].astype(float) * 1.0
ranked["score"] += ranked["novelty_distance"].fillna(0.0)

if "critic_v_error" in ranked.columns:
    ranked["score"] += (ranked["critic_v_error"].abs() <= 1).astype(float)
    ranked["score"] -= 0.25 * ranked["critic_v_error"].abs()

print("Top 10 generated routes by composite score:")
print("=" * 70)
top_routes = ranked.sort_values("score", ascending=False).head(10)
display_cols = ["board_key", "score", "basic_valid_eval", "strict_valid_eval", "novelty_distance"]
if "critic_v_error" in top_routes.columns:
    display_cols.append("critic_v_error")
print(top_routes[display_cols].to_string(index=False))

## Save evaluation results

We save the full evaluation DataFrame and the top candidates for further analysis.



In [ ]:
# Save evaluation results
OUT_DIR = ROOT / "data" / "processed" / "evaluation"
OUT_DIR.mkdir(parents=True, exist_ok=True)

df_eval.to_csv(OUT_DIR / "generated_route_evaluation.csv", index=False)
top_candidates = ranked.sort_values("score", ascending=False).head(100)
top_candidates.to_csv(OUT_DIR / "top_generated_candidates.csv", index=False)

print(f"Saved evaluation results to: {OUT_DIR}")
print(f"  - generated_route_evaluation.csv ({len(df_eval)} rows)")
print(f"  - top_generated_candidates.csv (100 rows)")

print("\n" + "=" * 50)
print("EVALUATION SUMMARY")
print("=" * 50)
print(f"\nTotal generated routes: {len(df_eval):,}")
print(f"\nBasic validity rate: {df_eval['basic_valid_eval'].mean():.1%}")
print(f"Strict validity rate: {df_eval['strict_valid_eval'].mean():.1%}")
print(f"Mean novelty distance: {df_eval['novelty_distance'].mean():.3f}")

if 'critic_v_error' in df_eval.columns:
    print(f"\nGrade consistency:")
    print(f"  Exact V-grade: {(df_eval['critic_v_error'] == 0).mean():.1%}")
    print(f"  Within 1 V-grade: {(df_eval['critic_v_error'].abs() <= 1).mean():.1%}")
    print(f"  Within 2 V-grades: {(df_eval['critic_v_error'].abs() <= 2).mean():.1%}")
else:
    print("\n(Grade consistency not available - no critic model loaded)")

## Key Takeaways

1. **Validity**: The generator produces routes that mostly satisfy structural constraints (start/finish holds, no duplicates, single board).

2. **Novelty**: Generated routes are meaningfully different from existing routes, as measured by Jaccard distance.

3. **Geometric plausibility**: The geometric features (height, width, hand reach) should be in reasonable ranges compared to real routes.

4. **Grade consistency**: If the critic is available, we can check whether routes generated at a requested grade actually feel like that grade.

### Limitations

- Validity checks are structural, not semantic. A route might have valid start/finish holds but still be impossible.
- Geometric features are simple. More sophisticated analysis could check reachability and move sequences.
- The critic model was trained on real data, so it may not generalize well to novel route structures.

